In [ ]:
"""
1. Carregar
2. Tratar valores nulos "disfarçados"
3. Limpar texto (espaços, maiúsculas/minúsculas)
4. Corrigir encoding
5. Corrigir colunas numéricas sujas de posse de bola e precisao de passes.
6. Tratar outliers
7. Remover duplicatas
8. Reordenar colunas e ordenar linhas
9. Salvar
"""

In [ ]:
#carregamento de dados brutos e bagunçados
import pandas as pd
import numpy as np 

df = pd.read_csv(r"D:\projetos git\analise-de-times-br\data\raw\campeonato-brasileiro-estatisticas-full.csv")
display(df)

In [ ]:
#limpar dados nulos disfarçados
fake_nulls = ["", " ", "NA", "N/A", "na", "n/a", "null", "NULL", "None", "none"]
df.replace(fake_nulls, np.nan, inplace=True)



In [ ]:
#limpar texto de maisculas e minusculas e limpar espaços em branco
coluna_clube = df['clube']
for coluna in coluna_clube:
    df["clube"] = df["clube"].str.strip().str.title()

display(df)    

In [ ]:
#Correção do encoding do arquivo, o arquivo original esta com enconding incorreto
#Se faz necessário corrigir o encoding para evitar problemas com caracteres especiais
def corrigur_encoding(valor):
    if not isinstance(valor, str):
        return valor
    try:
        return valor.encode('latin1').decode('utf-8')
    except (UnicodeDecodeError, UnicodeEncodeError):
        return valor

for coluna in ['clube']:
    df[coluna] = df[coluna].apply(corrigur_encoding)

In [ ]:
#correção da coluna de posse de de bola, foi retirado o sinal de poprcetagem afim de potencialmente analisar estes dados mais tarde.
for coluna in ['posse_de_bola',"precisao_passes"]:
    df[coluna] = df[coluna].str.replace("%","", regex=False)
    df[coluna] = df[coluna].str.replace(" un","", regex=False)
    df[coluna] = df[coluna].str.replace(",",".", regex=False)
    df[coluna] = pd.to_numeric(df[coluna], errors= "coerce")

 #correção da coluna chutes e passe (so foram transformados para numericos)
for coluna in ["chutes","passes"]:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

df["chutes"] = df["chutes"].astype("Int64")
df["passes"] = df["passes"].astype("Int64")   

In [ ]:
#tratamento de outliers
#obs: tive um erro na logica do for ja qque inverti os sinais de < e >, agora esta funcionando 
#tanto cartoes veremlhos e amarelos nao podem ser negativos e alem disso tirei a casa decimal dos dois
for coluna in ["posse_de_bola","precisao_passes"]:
    fora_do_intervalo = (df[coluna] < 0) | (df[coluna] > 100)
    df.loc[fora_do_intervalo, coluna] = np.nan

cartao_amarelo_negativo = df["cartao_amarelo"] < 0 
df.loc[cartao_amarelo_negativo, "cartao_amarelo"] = np.nan  

cartao_vermelho_negativo = df["cartao_vermelho"] < 0 
df.loc[cartao_vermelho_negativo, "cartao_vermelho"] = np.nan  


df["cartao_amarelo"] = df["cartao_amarelo"].astype("Int64")
df["cartao_vermelho"] = df["cartao_vermelho"].astype("Int64")


In [ ]:
#removendo duplicatas
chave_unica = ["partida_id", "clube"]
duplicatas = df.duplicated(subset=chave_unica).sum()
df = df.drop_duplicates(subset=chave_unica, keep='first')
display(df)

In [ ]:
#ordenação da tabela de acordo com o id da partida e clube
ordem_das_colunas = [
    "partida_id", "rodata", "clube",
    "chutes", "chutes_no_alvo", "posse_de_bola", "precisao_passes", "passes",
    "escanteios", "faltas", "impedimentos",
    "cartao_amarelo", "cartao_vermelho",
]
df = df[ordem_das_colunas]
df = df.sort_values(["partida_id", "clube"]).reset_index(drop=False)
display(df)


In [ ]:
#salvar no caminho indicado
print("\n" + "=" * 60)
print("RELATÓRIO DE COBERTURA DE DADOS")
print("=" * 60)
 
colunas_para_checar = ["posse_de_bola", "precisao_passes", "chutes", "passes"]
 
for coluna in colunas_para_checar:
    total_preenchido = df[coluna].notna().sum()
    percentual_cobertura = (total_preenchido / len(df)) * 100
 
    # partida_id mínimo entre as linhas ONDE essa coluna tem dado
    linhas_com_dado = df[df[coluna].notna()]
    if len(linhas_com_dado) > 0:
        primeiro_id_com_dado = linhas_com_dado["partida_id"].min()
    else:
        primeiro_id_com_dado = "nenhum dado disponível"
 
    print(f"\n{coluna}:")
    print(f"  Cobertura: {total_preenchido}/{len(df)} linhas ({percentual_cobertura:.1f}%)")
    print(f"  Primeiro partida_id com dado preenchido: {primeiro_id_com_dado}")
 
print("\n" + "=" * 60)


caminho = r"D:\projetos git\analise-de-times-br\data\processed\campeonato-brasileiro-estatisticas-full.csv"
df.to_csv(caminho, index=False)